In [2]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np


In [8]:
def analyze_drift_log_with_ground_truth(file_path, buffer_size=30):
    """
    Reads the BLS drift log CSV and plots the data, 
    showing the Active Model Index, the Ground Truth Mode, and Buffer Cycles.
    """
    try:
        df = pd.read_csv(file_path)
    except FileNotFoundError:
        print(f"Error: Log file not found at {file_path}")
        return

    # --- Setup the Plot ---
    fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(12, 8), sharex=True, 
                                   gridspec_kw={'height_ratios': [3, 1]})
    
    # 1. Get points where the Data Set Mode changes (to separate batches)
    mode_change_indices = df[df['Data_Set_Mode'].diff() != 0].index.tolist()
    
    # 2. Get points where the Buffer is full (every 30 samples)
    buffer_indices = np.arange(buffer_size, df['Sample_Global'].max(), buffer_size)
    
    # --- Plot 1: Prediction vs. Target ---
    ax1.plot(df['Sample_Global'], df['Real_Target'], label='Real Target ($y$)', color='blue', linewidth=1.5)
    ax1.plot(df['Sample_Global'], df['Predicted_Value'], label='Predicted Value ($\\hat{y}$)', color='red', linestyle='--', linewidth=1)
    
    # Highlight high error samples
    high_error_threshold = df['Error_Abs'].quantile(0.95)
    high_error_samples = df[df['Error_Abs'] > high_error_threshold]
    # ax1.scatter(high_error_samples['Sample_Global'], high_error_samples['Predicted_Value'], 
                # color='orange', marker='o', s=20, label='High Error Samples')

    ax1.set_title('BLS Model Prediction vs. Ground Truth Over Time (Buffer Cycles Highlighted)', fontsize=14)
    ax1.set_ylabel('Value', fontsize=12)
    ax1.legend(loc='lower left')
    # ax1.grid(True, linestyle=':', alpha=0.6)
    
    # --- Plot 2: Active Model Index & Ground Truth ---
    
    
    ax2.plot(df['Sample_Global'], df['Data_Set_Mode'], drawstyle='steps-post', 
             label='Ground Truth Mode (Data Set)', color='purple', linestyle='-', linewidth=3, alpha=0.6)
    
    
    ax2.plot(df['Sample_Global'], df['Active_Model_Index'], drawstyle='steps-post', 
             label='Chosen Model (Drift Detector Output)', color='green', linewidth=2)
    
    mode_indices = df['Data_Set_Mode'].unique() # Use data set mode for y-axis range
    ax2.set_yticks(sorted(mode_indices))
    ax2.set_yticklabels([f'Mode {int(i)}' for i in sorted(mode_indices)])
    
    ax2.set_title('Model Tracking: Chosen Model vs. Ground Truth', fontsize=14)
    ax2.set_xlabel('Global Sample Index (Time)', fontsize=12)
    ax2.set_ylabel('Model Index', fontsize=12)
    ax2.legend(loc='lower right')
    ax2.grid(True, linestyle=':', alpha=0.6)
    
    # --- Add Vertical Lines to Both Plots ---
    
    # Add Buffer Cycle Lines (Dotted Black)
    for buffer_idx in buffer_indices:
        ax1.axvline(x=buffer_idx, color='black', linestyle=':', linewidth=0.7, alpha=0.5)
        ax2.axvline(x=buffer_idx, color='black', linestyle=':', linewidth=0.7, alpha=0.5)

    # Add Data Set Change Lines (Solid Gray)
    for mode_idx in mode_change_indices:
        ax1.axvline(x=mode_idx, color='gray', linestyle='-', linewidth=1.5, alpha=0.7)
        ax2.axvline(x=mode_idx, color='gray', linestyle='-', linewidth=1.5, alpha=0.7)
        
        # current_data_mode = df.loc[mode_idx, 'Data_Set_Mode']
        # if current_data_mode != 0:
        #      mode_text = f"Drift Starts (Data Set {current_data_mode})"
        #      ax1.text(mode_idx + 2, ax1.get_ylim()[1] * 0.95, mode_text, rotation=0, 
        #               verticalalignment='top', fontsize=10, color='gray')


    # Final adjustments and saving
    plt.tight_layout()
    plot_filename = 'bls_drift_analysis_fp_r2b.png'
    plt.savefig(plot_filename)
    plt.close()
    
    print(f"Analysis saved to {plot_filename}")

# Execute the function with the assumed buffer size of 30
analyze_drift_log_with_ground_truth('newIdea/bls_drift_log_fp_r2b.csv', buffer_size=30)

Analysis saved to bls_drift_analysis_fp_r2b.png
